In [3]:
import bpmeth
import sympy as sp
import re

def format_derivative(expr):
    if isinstance(expr, sp.Derivative):
        # Get the function and its name
        f = expr.expr
        name = f.func.__name__  # e.g., 'a1'
        index_match = re.match(r'([ab]+)([\ds]+)', name)
        if index_match:
           base, idx = index_match.groups()
           prime = "'" * expr.derivative_count
           return sp.var(f"{base}{prime}_{idx}")
        else:
            return expr
    elif isinstance(expr, sp.Function):
        name = expr.func.__name__
        return sp.var(name)
    return expr

def format_function(expr):
    if isinstance(expr, sp.Function):
        name = expr.func.__name__
        index_match = re.match(r'([ab]+)([\ds]+)', name)
        if index_match:
           base, idx = index_match.groups()
           return sp.var(f"{base}_{idx}")        
        return sp.var(name)
    return expr



## Field derivatives expansion with curvature

The expansion is based on this scalar potential definition

$$
\begin{align*}
\phi(x,y,s) &= \sum_{i=0}^{\infty}\phi_i(x,s)\frac{y^i}{i!}\\
\end{align*}
$$
for which the laplace equation gives:

$$
\phi_{i+2} = -\frac{1}{1 + hx} \left( \partial_x \left( (1 + hx) \partial_x \phi_i \right) + \partial_s \left( \frac{1}{1 + hx} \partial_s \phi_i \right) \right)
$$

Defining $\phi_0(x,s)$ and $\phi_1(x,s)$ we get

$$
\begin{align*}
\phi_0(x,s) &= -a_0(s)-\sum_{n=1}^{\infty}a_n(s)\frac{x^n}{n!} &
\phi_1(x,s) &= -\sum_{n=1}^{\infty}b_n(s)\frac{x^{n-1}}{(n-1)!} \\
a_n(s) &= \left. \partial_x^{n-1} B_x(x, y, s) \right|_{x = y = 0} &
b_n(s) &= \left. \partial_x^{n-1} B_y(x, y, s) \right|_{x = y = 0}
a_0(s)
&& a_0'(s)=b_s(s) = B_s(0, 0, s)
\end{align*}
$$

In [2]:
order=2 # in the transverse field derivatives  [n]
nphi=1 # order of the scalar potential components [i]

s,h=sp.var("s,h", real=True)
aa = [sp.Function(f"a{i+1}")(s) for i in range(order)]
bb = [sp.Function(f"b{i+1}")(s) for i in range(order)]
bs = sp.Function("b_s")(s)

fm=bpmeth.FieldExpansion(a=aa,b=bb,bs=bs,hs=h,nphi=nphi)

bxys=fm.get_Bfield(lambdify=False,subs=False)
def simpl(bb):
    bb=bb.series(h,0,2).removeO().expand()
    bb=bb.replace(lambda e: isinstance(e, sp.Derivative),format_derivative)
    bb=bb.replace(lambda e: isinstance(e, sp.Function),format_function)
    return bb
    
    
bx,by,bs=map(simpl,bxys)
bxd=bx.as_coefficients_dict(fm.x,fm.y)
byd=by.as_coefficients_dict(fm.x,fm.y)
bsd=bs.as_coefficients_dict(fm.x,fm.y)
out=[sp.var('term,B_x,B_y,B_s')]
terms=set(bxd)|set(byd)|set(bsd)
key=sp.polys.orderings.monomial_key('grlex', (fm.y, fm.x))
for ll in sorted(terms,key=key):
    out.append([ll,bxd[ll],byd[ll],bsd[ll]])
sp.Matrix(out)

Matrix([
[  term, B_x, B_y,              B_s],
[     1, a_1, b_1,              b_s],
[     x, a_2, b_2,     a'_1 - b_s*h],
[     y, b_2,   0,             b'_1],
[  x**2,   0,   0, -a'_1*h + a'_2/2],
[   x*y,   0,   0,   -b'_1*h + b'_2],
[  x**3,   0,   0,        -a'_2*h/2],
[x**2*y,   0,   0,          -b'_2*h]])

 We have to be careful: limiting in nphi also limits in terms in h!

## Field derivatives expansion without curvature

In [70]:
order=6 # in the transverse field derivatives  [n]
nphi=5 # order of the scalar potential components [i]

s,h=sp.var("s,h", real=True)
aa = [sp.Function(f"a{i+1}")(s) for i in range(order)]
bb = [sp.Function(f"b{i+1}")(s) for i in range(order)]
bs = sp.Function("b_s")(s)

fm=bpmeth.FieldExpansion(a=aa,b=bb,bs=bs,hs="0",nphi=nphi)

bxys=fm.get_Bfield(lambdify=False,subs=False)
def simpl(bb):
    bb=bb.series(h,0,2).removeO().expand()
    bb=bb.replace(lambda e: isinstance(e, sp.Derivative),format_derivative)
    bb=bb.replace(lambda e: isinstance(e, sp.Function),format_function)
    return bb
    
    
bx,by,bs=map(simpl,bxys)
bxd=bx.as_coefficients_dict(fm.x,fm.y)
byd=by.as_coefficients_dict(fm.x,fm.y)
bsd=bs.as_coefficients_dict(fm.x,fm.y)
out=[sp.var('term,B_x,B_y,B_s')]
terms=set(bxd)|set(byd)|set(bsd)
key=sp.polys.orderings.monomial_key('grlex', (fm.y, fm.x))
for ll in sorted(terms,key=key):
    out.append([ll,bxd[ll],byd[ll],bsd[ll]])
sp.Matrix(out)

Matrix([
[     term,                            B_x,                           B_y,                               B_s],
[        1,                            a_1,                           b_1,                               b_s],
[        x,                            a_2,                           b_2,                              a'_1],
[        y,                            b_2,                   -a_2 - b'_s,                              b'_1],
[     x**2,                          a_3/2,                         b_3/2,                            a'_2/2],
[      x*y,                            b_3,                  -a''_1 - a_3,                              b'_2],
[     y**2,               -a''_1/2 - a_3/2,              -b''_1/2 - b_3/2,                 -a'_2/2 - b''_s/2],
[     x**3,                          a_4/6,                         b_4/6,                            a'_3/6],
[   x**2*y,                          b_4/2,              -a''_2/2 - a_4/2,                            b

In [45]:
bs=-0.03*s**4 + 0.04*s**3
bs

-0.03*s**4 + 0.04*s**3

In [36]:
# terms Bx:x, By:y 
a2=-bs.diff(s)/2;
sp.Matrix([a2,-a2-bs.diff(s)])

Matrix([
[0.06*s**3 - 0.06*s**2],
[0.06*s**3 - 0.06*s**2]])

In [42]:
# terms Bs:x**2+y**2
sp.Matrix([a2.diff(s)/2, (-a2.diff(s)/2-bs.diff(s,2)/2).expand()])

Matrix([
[0.09*s**2 - 0.06*s],
[0.09*s**2 - 0.06*s]])

In [63]:
#terms By:y*r**2
a4=sp.var("a_4")
a4=sp.solve(-a2.diff(s,2)/2 - a4/2 -(a2.diff(s,2)/3+a4/6+bs.diff(s,3)/6),a4)[0]
a4

0.09 - 0.27*s

In [64]:
sp.Matrix([a4/6, -a2.diff(s,2)/2 - a4/2])

Matrix([
[0.015 - 0.045*s],
[0.015 - 0.045*s]])

In [66]:
#terms Bs: r**4
sp.Matrix([a4.diff(s)/24, a2.diff(s,3)/12+a4.diff(s)/24+bs.diff(s,4)/24])

Matrix([
[-0.01125],
[-0.01125]])

In [69]:
-a2.diff(s,3)/4-a4.diff(s)/4

-0.0225000000000000

In [5]:
x,y,s=sp.var("x,y,s")

bs = -0.03*s**4 + 0.04*s**3
b1 = "0.0"  # k0
b2 = "0.0"  # k1
b3 = "0.0"  # k2
a1 = "0.0"  # ks0
#a2 = -eval(bs, {"s": A_magnet_entry.s}).diff(A_magnet_entry.s) / 2  # ks1
a2 = 0.06 * s**3 - 0.06 * s**2  # ks1
a3 = "0.0"  # ks2
a4 = -0.27 * s + 0.09  # ks1
h = "0.0"
length = 2
A_magnet_entry = bpmeth.GeneralVectorPotential(
    hs=h, b=(b1, b2, b3), a=(a1, a2, a3, a4), bs=bs, nphi=10
)
x = A_magnet_entry.x
y = A_magnet_entry.y
s = A_magnet_entry.s


In [6]:
Bx_sp, By_sp, Bs_sp = A_magnet_entry.get_Bfield(lambdify=False)

In [7]:
sp.Matrix([Bx_sp, By_sp, Bs_sp])

Matrix([
[                                          x**3*(0.09 - 0.27*s)/6 + x*(0.06*s**3 - 0.06*s**2) - y**2*(24*x*(0.09 - 0.27*s) + 24*x*(0.36*s - 0.12))/48],
[                                        y**3*(2.16 - 6.48*s)/144 - y*(-1.44*s**3 + 1.44*s**2 + 12*x**2*(0.09 - 0.27*s) + 12*x**2*(0.36*s - 0.12))/24],
[-0.03*s**4 + 0.04*s**3 - 0.01125*x**4 + x**2*(0.18*s**2 - 0.12*s)/2 - 0.01125*y**4 - y**2*(4.32*s**2 + 24*s*(0.24 - 0.36*s) - 2.88*s + 1.08*x**2)/48]])

In [8]:
(Bx_sp.diff(x) + By_sp.diff(y) + Bs_sp.diff(s)).simplify()

0

In [9]:
list(
    map(
        sp.simplify,
        (
            sp.diff(Bs_sp, y) - sp.diff(By_sp, s),
            sp.diff(Bx_sp, s) - sp.diff(Bs_sp, x),
            sp.diff(By_sp, x) - sp.diff(Bx_sp, y),
        ),
    )
)


[0, 0, 0]